### Data Ingestion-loading the data and converting it to document structure

In [11]:
# ! pip install -r requirements.txt

In [2]:
from langchain_community.document_loaders import PyMuPDFLoader

file_path = "data/ai_notes.pdf"
loader = PyMuPDFLoader(file_path)
doc=loader.load()
print(doc[0:10])


[Document(metadata={'producer': 'Skia/PDF m119', 'creator': 'Chromium', 'creationdate': '2024-05-23T06:52:48+00:00', 'source': 'data/ai_notes.pdf', 'file_path': 'data/ai_notes.pdf', 'total_pages': 187, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-05-23T06:52:48+00:00', 'trapped': '', 'modDate': "D:20240523065248+00'00'", 'creationDate': "D:20240523065248+00'00'", 'page': 0}, page_content='Artificial Intelligence\n1\nArtificial Intelligence\nUnit - I\nAI Definition\nTypes of Artificial Intelligence:\nProblems\nThe foundations of Artificial Intelligence\nTechniques\nModels\nDefining Problem as a state space search\nState Space Search\nFeatures of State Space Search\nSteps in State Space Search\nState Space Representation\nExample of State Space Search\nApplications of State Space Search\nProduction system\nIntelligent Agents: Agents and Environments\n1. Intelligent Agents:\n2. Agents and Environments:\nExamples:\nCharacteristics\nTypes o

In [3]:
def load_documents():
    loader=PyMuPDFLoader(file_path)
    doc=loader.load()
    return doc

### document parsing-Chunking

In [11]:
def is_useful_chunk(text: str, min_length: int = 200) -> bool:
    lines = text.strip().split('\n')
    non_empty_lines = [l for l in lines if l.strip()]

    # Reject if too short
    if len(text.strip()) < min_length:
        return False

    # Reject if most lines are very short (TOC pattern)
    short_lines = [l for l in non_empty_lines if len(l.strip()) < 60]
    if len(non_empty_lines) > 0 and (len(short_lines) / len(non_empty_lines)) > 0.75:
        return False

    return True

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

print(type(doc))
print(len(doc))
text_splitter = RecursiveCharacterTextSplitter(
    # Set a really small chunk size, just to show.
    chunk_size=1000,
    chunk_overlap=200,
  )
texts = text_splitter.split_documents(doc)
filtered_Chunks= [doc for doc in texts if is_useful_chunk(doc.page_content)]
print(len(texts))
for i in range(4):
    print(f"Chunk {i}:",filtered_Chunks[i])


<class 'list'>
187
364
Chunk 0: page_content='Artificial Intelligence
4
Unit - I
AI Definition
Artificial Intelligence (AI) refers to the simulation of human intelligence processes 
by computer systems. These processes include learning (the acquisition of 
information and rules for using the information), reasoning (using rules to reach 
approximate or definite conclusions), and self-correction. AI systems are designed 
to perform tasks that normally require human intelligence, such as visual 
perception, speech recognition, decision-making, and language translation. The 
goal of AI is to develop machines that can think, learn, and adapt like humans, 
ultimately enhancing efficiency, productivity, and innovation across various 
industries.
Types of Artificial Intelligence:
Working of Min-Max Algorithm:
Alpha-Beta Cut-Offs
Working of Alpha-Beta Pruning:
Move Ordering in Alpha-Beta pruning:
Rules to find good ordering:
Natural Language Processing (NLP)
Learning
Explanation-based Learning

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
def chunking(documents):
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )
    chunks=text_splitter.split_documents(documents)
    filtered_Chunks= [doc for doc in chunks if is_useful_chunk(doc.page_content)]

    print("-------------------Chunking has been done!----------------------------------")
    return filtered_Chunks

### Creating vectors and storing them in VectorDB-Chroma (can use fiass or pinecone too)

In [14]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [15]:
text = "This is a test document."
query_result = embeddings.embed_query(text)
print(query_result)

[-0.04895181208848953, -0.03986204415559769, -0.02156280353665352, 0.009908564388751984, -0.03810390830039978, 0.01268436387181282, 0.04349453002214432, 0.07183387130498886, 0.00974862277507782, -0.006987064611166716, 0.06352808326482773, -0.030322642996907234, 0.013839490711688995, 0.025805894285440445, -0.0011362676741555333, -0.014563605189323425, 0.04164033755660057, 0.03622833266854286, -0.0268008504062891, 0.025120766833424568, -0.024978570640087128, -0.004533296450972557, -0.0266671534627676, 0.004100735764950514, -0.052048034965991974, -0.009930513799190521, -0.052065297961235046, 0.008992106653749943, -0.03830047696828842, -0.04405849799513817, -0.004204395227134228, 0.07047972083091736, 0.005133919417858124, -0.07161542773246765, 1.697531274658104e-06, -0.00604770565405488, -0.011076392605900764, 0.017513373866677284, -0.0222998708486557, 0.04095493629574776, 0.033790186047554016, 0.05665040388703346, -0.07114939391613007, 0.04097664728760719, -0.005906015168875456, -0.032970

In [16]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
def makeVectors_storeDB(chunks):
    embedding_model=HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-mpnet-base-v2"
    )


    vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory='chroma_langchain_db'
    )
    return vectorstore

In [17]:
if __name__=="__main__":
    docs=load_documents()
    chunks=chunking(docs)
    vec_db=makeVectors_storeDB(chunks)
    print("-------------------Ingestion Pipeline has been completed!---------------------------")

-------------------Chunking has been done!----------------------------------


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

-------------------Ingestion Pipeline has been completed!---------------------------
